## Dataset preparation

Use **Run All** to inspect the dataset and convert its dates. The date-conversion cell can also run independently. Both loading cells support kernels started from the project root or the `notebooks` folder.

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

path = project_root / "data" / "raw" / "Primordic_IssueDupesLite.csv"

df = pd.read_csv(path)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head(5))

print("\nMissing values:")
print(df.isna().sum())



Shape: (10000, 13)

Columns: ['name', 'owner', 'repo', 'primary_issue_num', 'primary_title', 'primary_body', 'primary_issue_created_at', 'duplicate_issue_num', 'duplicate_title', 'duplicate_body', 'duplicate_issue_created_at', 'duplicate_closing_comment', 'comment_created_at']


,name,owner,repo,primary_issue_num,primary_title,primary_body,primary_issue_created_at,duplicate_issue_num,duplicate_title,duplicate_body,duplicate_issue_created_at,duplicate_closing_comment,comment_created_at
0,1kc/razer-macos,1kc,razer-macos,107,Huntsman Mini support,"Great job building this, first of all.\nMy Hun...",2020-11-30T12:23:31Z,286,Razer Huntsman mini wont work,Please add support for huntsman mini iam waiti...,2021-03-09T17:30:06Z,Huntsman Mini progress is tracked in #107.,2021-03-09T21:37:52Z
1,1kc/razer-macos,1kc,razer-macos,497,Support for Razer Basilisk X Hyperspeed,"Would love to see support for this mouse, thanks!",2021-11-23T07:40:07Z,543,Support for Razer Basilisk X Hyperspeed,I was wondering if it would be possible to get...,2022-01-13T03:14:20Z,Duplicate of #497. I did add support though. T...,2022-01-14T20:06:49Z
2,1kc/razer-macos,1kc,razer-macos,309,Please Philips Hue support,as title says 😸,2021-03-18T11:43:57Z,398,Chroma Connect Module,Is it possible to emulate the Chroma Connect M...,2021-06-12T09:52:43Z,Duplicate of #309,2021-06-22T19:28:42Z
3,1kc/razer-macos,1kc,razer-macos,464,Support for Basilisk V3,NaN,2021-10-15T00:38:11Z,557,Razer Basilisk V3 not working,Hi does the latest version support the Basilis...,2022-01-27T22:51:05Z,"No, but the next version should. The backend w...",2022-01-27T22:55:28Z
4,1kc/razer-macos,1kc,razer-macos,250,Huntsman v2 Analog Support,Let me know if there is other Huntsman v2 supp...,2021-02-16T05:33:46Z,320,Razer Huntsman V2 Analog support,The lastest version 0.3.9 does not recognize t...,2021-03-21T22:59:59Z,Duplicate of #250\nLet's track the progress th...,2021-03-22T10:42:30Z



Missing values:
name                            0
owner                           0
repo                            0
primary_issue_num               0
primary_title                   0
primary_body                  175
primary_issue_created_at        0
duplicate_issue_num             0
duplicate_title                 0
duplicate_body                217
duplicate_issue_created_at      0
duplicate_closing_comment       0
comment_created_at              0
dtype: int64


In [6]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

path = project_root / "data" / "raw" / "Primordic_IssueDupesLite.csv"
df = pd.read_csv(path)

df["primary_issue_created_at"] = pd.to_datetime(
    df["primary_issue_created_at"],
    utc=True,
    errors="coerce"
)

df["duplicate_issue_created_at"] = pd.to_datetime(
    df["duplicate_issue_created_at"],
    utc=True,
    errors="coerce"
)

print("Primary issue date range:")
print(
    df["primary_issue_created_at"].min(),
    df["primary_issue_created_at"].max()
)

print("\nDuplicate issue date range:")
print(
    df["duplicate_issue_created_at"].min(),
    df["duplicate_issue_created_at"].max()
)

print("\nPrimary issues created after their duplicates:")
print(
    (
        df["primary_issue_created_at"]
        > df["duplicate_issue_created_at"]
    ).sum()
)

Primary issue date range:
2008-07-11 12:40:56+00:00 2025-05-05 17:41:03+00:00

Duplicate issue date range:
2008-09-20 18:38:53+00:00 2025-05-06 18:02:37+00:00

Primary issues created after their duplicates:
1529


In [8]:

df_clean = df.copy()

text_columns = [
    "primary_title",
    "primary_body",
    "duplicate_title",
    "duplicate_body"
]

df_clean[text_columns] = df_clean[text_columns].fillna("")


df_clean["primary_text"] = (
    df_clean["primary_title"].str.strip()
    + "\n"
    + df_clean["primary_body"].str.strip()
).str.strip()

df_clean["duplicate_text"] = (
    df_clean["duplicate_title"].str.strip()
    + "\n"
    + df_clean["duplicate_body"].str.strip()
).str.strip()


print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("Empty primary texts:",
      df_clean["primary_text"].eq("").sum())

print("Empty duplicate texts:",
      df_clean["duplicate_text"].eq("").sum())

display(df_clean[["primary_text", "duplicate_text"]].head())

Original shape: (10000, 13)
Cleaned shape: (10000, 15)
Empty primary texts: 0
Empty duplicate texts: 0


,primary_text,duplicate_text
0,Huntsman Mini support\nGreat job building this...,Razer Huntsman mini wont work\nPlease add supp...
1,Support for Razer Basilisk X Hyperspeed\nWould...,Support for Razer Basilisk X Hyperspeed\nI was...
2,Please Philips Hue support\nas title says 😸,Chroma Connect Module\nIs it possible to emula...
3,Support for Basilisk V3,Razer Basilisk V3 not working\nHi does the lat...
4,Huntsman v2 Analog Support\nLet me know if the...,Razer Huntsman V2 Analog support\nThe lastest ...


In [9]:
df_clean["primary_id"] = (
    df_clean["name"].astype(str)
    + "#"
    + df_clean["primary_issue_num"].astype(str)
)

df_clean["duplicate_id"] = (
    df_clean["name"].astype(str)
    + "#"
    + df_clean["duplicate_issue_num"].astype(str)
)

print(
    "Unique primary issues:",
    df_clean["primary_id"].nunique()
)

print(
    "Unique duplicate issues:",
    df_clean["duplicate_id"].nunique()
)

print(
    "Unique issues overall:",
    pd.concat([
        df_clean["primary_id"],
        df_clean["duplicate_id"]
    ]).nunique()
)

print(
    "Repeated primary issues:",
    df_clean["primary_id"].duplicated().sum()
)

print(
    "Repeated duplicate issues:",
    df_clean["duplicate_id"].duplicated().sum()
)

print(
    "Self-referencing issue pairs:",
    (df_clean["primary_id"] == df_clean["duplicate_id"]).sum()
)

Unique primary issues: 9312
Unique duplicate issues: 10000
Unique issues overall: 19137
Repeated primary issues: 688
Repeated duplicate issues: 0
Self-referencing issue pairs: 43


In [10]:
self_refs = df_clean[
    df_clean["primary_id"] == df_clean["duplicate_id"]
].copy()

print("Self-referencing pairs:", len(self_refs))

display(
    self_refs[
        [
            "name",
            "primary_issue_num",
            "duplicate_issue_num",
            "primary_title",
            "duplicate_title"
        ]
    ].head(10)
)

Self-referencing pairs: 43


,name,primary_issue_num,duplicate_issue_num,primary_title,duplicate_title
595,Figma-Linux/figma-linux,255,255,Font helper support (enhancement),Font helper support (enhancement)
838,JuliaGPU/CUDA.jl,2484,2484,Support for creating a CuSparseMatrixCSC from ...,Support for creating a CuSparseMatrixCSC from ...
901,KurtBestor/Hitomi-Downloader,4181,4181,hitomi not downloading,hitomi not downloading
1025,Masterminds/glide,276,276,Error with golang.org/x/text,Error with golang.org/x/text
1035,Mastermindzh/tidal-hifi,329,329,Build for the raspberry pi,Build for the raspberry pi
2019,aframevr/aframe,675,675,Consider supporting Bower,Consider supporting Bower
2257,ant-design/ant-design-pro,187,187,何时可以提供Typescript版本撒？,何时可以提供Typescript版本撒？
2331,apache/pinot,9625,9625,[multistage] [stability] Clean up MailboxServi...,[multistage] [stability] Clean up MailboxServi...
2673,bcherny/json-schema-to-typescript,476,476,Generated types of referenced schemas are not ...,Generated types of referenced schemas are not ...
2907,c3js/c3,2305,2305,Bar char drops an arbitrary data series when x...,Bar char drops an arbitrary data series when x...


In [11]:
print(
    "Identical titles:",
    (
        self_refs["primary_title"]
        == self_refs["duplicate_title"]
    ).sum()
)

print(
    "Identical combined texts:",
    (
        self_refs["primary_text"]
        == self_refs["duplicate_text"]
    ).sum()
)

Identical titles: 43
Identical combined texts: 43


In [12]:
df_clean = df_clean[
    df_clean["primary_id"] != df_clean["duplicate_id"]
].copy()

df_clean.reset_index(drop=True, inplace=True)


print("Original pairs:", len(df))
print("Removed self-references:", len(df) - len(df_clean))
print("Remaining valid pairs:", len(df_clean))

print(
    "Remaining self-references:",
    (df_clean["primary_id"] == df_clean["duplicate_id"]).sum()
)

Original pairs: 10000
Removed self-references: 43
Remaining valid pairs: 9957
Remaining self-references: 0


C:\Users\user\AppData\Local\Temp\ipykernel_13424\2178909582.py:5: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.reset_index(drop=True, inplace=True)


In [13]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "issuedupes_clean.csv"

df_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df_clean.shape)

Saved to: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\data\processed\issuedupes_clean.csv
Shape: (9957, 17)
